# DAPA: load public data

Fills the gold tables that Create Tables made, from the public data collected for **one metro**, and adds
5-year lifetime value per tract. Run Create Tables first.

Inputs are the files the deploy script uploads to `Files/dapa` in the lakehouse: the metro's tables from
`data/processed/<sample>/` and the two files in `data/assumptions/`. To switch metro, change `SAMPLE` in
`fabric/deploy/config.env` and deploy again. Nothing in this notebook names a region.

| Filled here | From |
|---|---|
| `dim_county`, `dim_tract`, `dim_year` | Census geography, ACS 5-year, FDIC household survey rates |
| `dim_bank`, `dim_branch`, `fact_branch_deposits`, `fact_branch_events` | FDIC BankFind |
| `fact_tract_lending` | HMDA |
| `fact_tract_complaints` | CFPB complaints, ZIP mapped to tract |
| `scenario_ltv`, `tract_ltv`, `ltv_assumptions`, `ltv_customer_year` | LTV assumptions file |

Left empty, because no pilot has run: `fact_tract_accounts`, `dim_event`, `fact_attendee`, `agg_playbook`.

Unbanked and underbanked counts are **estimates**, and every lifetime value figure is an **assumption**.
Run locally with plain Python and it reads the repo's `data/` folder and writes CSVs to `sample-output/`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

CALIBRATE_TO_METRO = True  # scale tract estimates so they add up to the FDIC survey's rate for this metro
LOW_INCOME_BRACKETS = ["lt_15k", "15k_30k"]  # household income under $30,000
YEARS = 5
SCENARIOS = {"conservative": "Conservative", "base": "Base", "optimistic": "Optimistic"}  # assumptions file -> dashboard label

LAKEHOUSE_FILES = Path("/lakehouse/default/Files/dapa")
IDS = {"tract_geoid": str, "county_fips": str, "zip": str, "zcta": str}


def read(name: str) -> pd.DataFrame:
    """The lakehouse copy in Fabric; locally, the repo copy for the DAPA_SAMPLE metro (default houston)."""
    if (LAKEHOUSE_FILES / f"{name}.csv").exists():
        return pd.read_csv(LAKEHOUSE_FILES / f"{name}.csv", dtype=IDS)
    root = Path(os.environ.get("DAPA_REPO", Path.cwd()))
    for folder in [root, *root.parents]:
        for sub in (f"data/processed/{os.environ.get('DAPA_SAMPLE', 'houston')}", "data/assumptions"):
            if (folder / sub / f"{name}.csv").exists():
                return pd.read_csv(folder / sub / f"{name}.csv", dtype=IDS)
    raise FileNotFoundError(f"{name}.csv: not in Files/dapa. Run the deploy script to upload it.")


def pct_rank(s: pd.Series, ascending: bool = True) -> pd.Series:
    """Score in (0, 1] by percentile rank, so no tract scores exactly zero."""
    return s.rank(pct=True, ascending=ascending).round(4)


def nearest(lat, lon, to_lat, to_lon):
    """For each point, the index of and distance in km to the nearest of the `to` points."""
    lat, lon, to_lat, to_lon = (np.radians(np.asarray(v, dtype=float)) for v in (lat, lon, to_lat, to_lon))
    a = (np.sin((to_lat[None, :] - lat[:, None]) / 2) ** 2
         + np.cos(lat[:, None]) * np.cos(to_lat[None, :]) * np.sin((to_lon[None, :] - lon[:, None]) / 2) ** 2)
    km = 6371 * 2 * np.arcsin(np.sqrt(a))
    return km.argmin(axis=1), km.min(axis=1)

## Banks and branches

Local-bank branches (`target_issuer`: assets under $10B) are the marketing opportunities.

In [ ]:
branches = read("branches")
dim_bank = (
    branches.groupby("cert", as_index=False)
    .agg(fed_rssd=("fed_rssd", "first"), bank_name=("bank_name", "first"), assets_000=("assets_000", "first"),
         target_issuer=("target_issuer", "first"), is_mdi=("is_mdi", "max"))
)
dim_bank["fed_rssd"] = dim_bank["fed_rssd"].astype("Int64").astype("string")
dim_bank["is_tenant_bank"] = False  # no bank tenant chosen yet
dim_bank["issuer_type"] = np.where(dim_bank["target_issuer"], "Local bank (under $10B)", "Large bank")

dim_branch = branches[["uninum", "cert", "tract_geoid", "office_name", "city", "latitude", "longitude", "servtype_desc"]].copy()
dim_branch["servtype_desc"] = dim_branch["servtype_desc"].str.title()

# Deposits are kept for branches open today, so every row joins to a branch on the map.
deposits = read("branch_deposits")[["uninum", "year", "branch_deposits_000"]]
fact_branch_deposits = deposits[deposits["uninum"].isin(dim_branch["uninum"])]

## Tracts: estimated unbanked households and opportunity score

**Estimate.** FDIC survey rates by income bracket are applied to the ACS household count in each bracket.
With `CALIBRATE_TO_METRO`, the result is scaled so the metro total matches the FDIC survey's own rate for this metro.

**Score.** Opportunity = need x bank fit, each a percentile rank within the metro.
Need ranks the estimated unbanked share. Bank fit ranks how close the nearest local-bank branch is.
Readiness to adopt is left empty: its inputs (broadband, smartphone access) are not collected yet.
Targeting uses place and need only (N5).

In [ ]:
acs = read("acs_tracts")
rates = read("fdic_rates_by_income")

dim_tract = acs[["tract_geoid", "county_fips", "tract_name"]].merge(
    read("tract_centroids")[["tract_geoid", "latitude", "longitude"]], on="tract_geoid", how="left")
dim_tract["households"] = acs["B19001_001E"]
bracket = {r.income_bracket: acs[[f"{v}E" for v in r.acs_variables.split()]].sum(axis=1) for r in rates.itertuples()}
dim_tract["low_income_households"] = sum(bracket[b] for b in LOW_INCOME_BRACKETS)
dim_tract["limited_english_households"] = acs[["C16002_004E", "C16002_007E", "C16002_010E", "C16002_013E"]].sum(axis=1)

survey = read("fdic_survey_rates")
metro = survey[(survey["geography_type"] == "msa") & (survey["year"] == survey["year"].max())
               & (survey["breakdown_var"] == "all") & survey["measure_var"].str.startswith("hbankstat")]
calibration = {}
for status, column in (("Unbanked", "unbanked"), ("Banked: Underbanked", "underbanked")):
    estimate = sum(bracket[r.income_bracket] * getattr(r, f"{column}_rate") for r in rates.itertuples())
    factor = 1.0
    if CALIBRATE_TO_METRO and (metro["category"] == status).any():
        metro_rate = metro.loc[metro["category"] == status, "percent"].iloc[0] / 100
        factor = metro_rate * dim_tract["households"].sum() / estimate.sum()
    calibration[column] = round(factor, 3)
    dim_tract[f"est_{column}_households"] = (estimate * factor).round(1)
print("calibration factors (1.0 = national rates as published):", calibration)

target = branches[branches["target_issuer"]]
_, km_to_local_branch = nearest(dim_tract["latitude"], dim_tract["longitude"], target["latitude"], target["longitude"])
unbanked_share = (dim_tract["est_unbanked_households"] / dim_tract["households"]).where(dim_tract["households"] > 0)
dim_tract["need_score"] = pct_rank(unbanked_share)
dim_tract["readiness_score"] = np.nan
dim_tract["bank_fit_score"] = pct_rank(pd.Series(km_to_local_branch), ascending=False)
dim_tract["opportunity_score"] = (dim_tract["need_score"] * dim_tract["bank_fit_score"]).round(4)
dim_tract["pilot_arm"] = "Not in pilot"

counties = read("sample_counties")
dim_county = counties[counties["county_fips"].isin(dim_tract["county_fips"])][["county_fips", "county_name", "state"]]

## Community evidence: branch changes, lending, complaints

- Branch events carry an address but no tract. A branch still open takes its tract from `dim_branch`;
  a closed one takes the tract with the nearest centre, which is an approximation.
- Complaints are published by ZIP. Each ZIP's count is split across its tracts by land-area share.

In [ ]:
events = read("branch_events")
events["event"] = events["event"].replace({"Branch Opening": "Opening", "Branch Closing": "Closing"})
events = events.merge(dim_branch[["uninum", "tract_geoid"]], on="uninum", how="left")
missing = events["tract_geoid"].isna() & events["latitude"].notna()
idx, _ = nearest(events.loc[missing, "latitude"], events.loc[missing, "longitude"], dim_tract["latitude"], dim_tract["longitude"])
events.loc[missing, "tract_geoid"] = dim_tract["tract_geoid"].to_numpy()[idx]
events["year"] = events["effective_date"].str[:4].astype(int)
fact_branch_events = events[["uninum", "tract_geoid", "year", "effective_date", "event"]]
# Keep only rows that join to a tract in this metro, so no chart or slicer shows an unmatched group.
dim_branch = dim_branch[dim_branch["tract_geoid"].isin(dim_tract["tract_geoid"])]
dim_bank = dim_bank[dim_bank["cert"].isin(dim_branch["cert"])]
fact_branch_deposits = fact_branch_deposits[fact_branch_deposits["uninum"].isin(dim_branch["uninum"])]
fact_branch_events = fact_branch_events[fact_branch_events["tract_geoid"].isin(dim_tract["tract_geoid"])]

lending = read("hmda_tract_year")
fact_tract_lending = lending[lending["loan_purpose_group"] == "all"][
    ["tract_geoid", "year", "applications", "originations", "denials", "high_cost_originations", "owner_occupied_units"]].round()  # counts converted from 2010 tracts are fractional
fact_tract_lending = fact_tract_lending[fact_tract_lending["tract_geoid"].isin(dim_tract["tract_geoid"])]

complaints = read("complaints_zip_year").groupby(["zip", "year", "product_group"], as_index=False)["complaints"].sum()
complaints = complaints.merge(read("zcta_tract")[["zcta", "tract_geoid", "zcta_share"]], left_on="zip", right_on="zcta")
complaints["complaints"] = complaints["complaints"] * complaints["zcta_share"]
fact_tract_complaints = (
    complaints[complaints["tract_geoid"].isin(dim_tract["tract_geoid"])]
    .groupby(["tract_geoid", "year", "product_group"], as_index=False)["complaints"].sum().round(3)
)

years = pd.concat([t["year"] for t in (fact_branch_deposits, fact_branch_events, fact_tract_lending, fact_tract_complaints)])
dim_year = pd.DataFrame({"year": sorted(years.unique())})

## Lifetime value of one new customer

Each year's revenue = debit interchange + deposit spread + fees, plus cross-sell revenue from the cross-sell start year.
A customer counts as active for all of year 1, so the share still active in year *t* is retention^(t-1).
Revenue is discounted back to today.

`scenario_ltv` holds the cumulative revenue per customer, before acquisition cost, so the dashboard can set it
against the measured cost per active customer. `ltv_customer_year` shows the working, net of acquisition cost.

In [ ]:
ltv_assumptions = read("ltv_assumptions")


def params(scenario: str) -> dict:
    rows = ltv_assumptions[ltv_assumptions["scenario"].isin(["all", scenario])]
    return dict(zip(rows["parameter"], rows["value"]))


rows = []
for scenario, label in SCENARIOS.items():
    p = params(scenario)
    interchange = p["debit_spend_per_year"] * p["interchange_rate"]
    deposit_spread = p["avg_deposit_balance"] * p["net_interest_margin"]
    cumulative = 0.0
    for year in range(1, YEARS + 1):
        cross_sell = p["cross_sell_probability"] * p["cross_sell_annual_revenue"] if year >= p["cross_sell_start_year"] else 0.0
        revenue = interchange + deposit_spread + p["annual_fees"] + cross_sell
        still_active = p["retention_rate"] ** (year - 1)
        cumulative += revenue * still_active / (1 + p["discount_rate"]) ** year
        rows.append({
            "scenario": label,
            "year": year,
            "interchange_usd": round(interchange, 2),
            "deposit_spread_usd": round(deposit_spread, 2),
            "fees_usd": round(p["annual_fees"], 2),
            "cross_sell_usd": round(cross_sell, 2),
            "revenue_usd": round(revenue, 2),
            "share_still_active": round(still_active, 4),
            "cumulative_ltv_usd": round(cumulative, 2),
            "acquisition_cost_usd": p["acquisition_cost"],
            "cumulative_net_ltv_usd": round(cumulative - p["acquisition_cost"], 2),
            "basis": "Assumption",
        })
ltv_customer_year = pd.DataFrame(rows)
scenario_ltv = ltv_customer_year[["scenario", "year", "cumulative_ltv_usd", "acquisition_cost_usd", "basis"]]
net_ltv = ltv_customer_year[ltv_customer_year["year"] == YEARS].set_index("scenario")["cumulative_net_ltv_usd"]

## Lifetime value per tract

Reachable households = estimated unbanked + estimated underbanked households.
Expected new customers = reachable households x adoption rate. Tract value = new customers x net LTV per customer.

In [ ]:
frames = []
for scenario, label in SCENARIOS.items():
    p = params(scenario)
    t = dim_tract[["tract_geoid"]].copy()
    t["scenario"] = label
    t["reachable_households"] = (dim_tract["est_unbanked_households"] + dim_tract["est_underbanked_households"]).round(1)
    t["new_customers"] = (t["reachable_households"] * p["adoption_rate"]).round(1)
    t["acquisition_spend_usd"] = (t["new_customers"] * p["acquisition_cost"]).round(0)
    t["net_value_5yr_usd"] = (t["new_customers"] * net_ltv[label]).round(0)
    frames.append(t)
tract_ltv = pd.concat(frames, ignore_index=True)

## Website snapshot

One JSON file with everything the ROI Engine web page shows, so the page never computes or invents a number.
Observed figures come from the tables above. The pilot is a **projection**: it picks a candidate local bank and
its branches from the data, then applies the rates in `pilot_assumptions.csv`.

In [ ]:
import json

pilot_p = dict(zip(*[read("pilot_assumptions")[c] for c in ("parameter", "value")]))
MILES = 1.609344
reachable = (dim_tract["est_unbanked_households"] + dim_tract["est_underbanked_households"]).to_numpy()
households = dim_tract["households"].sum()
local = dim_branch.merge(dim_bank[["cert", "bank_name", "target_issuer"]], on="cert")
local = local[local["target_issuer"]].merge(dim_tract[["tract_geoid", "opportunity_score"]], on="tract_geoid")

# Candidate pilot bank: the local bank whose branches sit in the highest-opportunity tracts.
n = int(pilot_p["pilot_branches"])
ranked = local.sort_values("opportunity_score", ascending=False).drop_duplicates(["cert", "tract_geoid"])
best = ranked.groupby("cert").filter(lambda g: len(g) >= 2 * n).groupby("cert")["opportunity_score"].apply(lambda s: s.head(2 * n).sum())
pilot_bank = ranked[ranked["cert"] == best.idxmax()].head(2 * n)
arms = {"pilot": pilot_bank.head(n), "control": pilot_bank.tail(n)}


def catchment(branches, exclude=()):
    _, km = nearest(dim_tract["latitude"], dim_tract["longitude"], branches["latitude"], branches["longitude"])
    inside = (km <= pilot_p["catchment_miles"] * MILES) & ~dim_tract["tract_geoid"].isin(exclude).to_numpy()
    return set(dim_tract["tract_geoid"][inside]), float(reachable[inside].sum())


pilot_tracts, pilot_reach = catchment(arms["pilot"])
_, control_reach = catchment(arms["control"], exclude=pilot_tracts)
opened = {"pilot": pilot_reach * pilot_p["control_uptake_90d"] * pilot_p["pilot_lift"],
          "control": control_reach * pilot_p["control_uptake_90d"]}
active = {"pilot": opened["pilot"] * pilot_p["pilot_activation_rate"],
          "control": opened["control"] * pilot_p["control_activation_rate"]}
# Compare like with like: scale the control arm to the pilot arm's catchment before taking the difference.
control_scale = pilot_reach / control_reach if control_reach else 0.0
incremental = active["pilot"] - active["control"] * control_scale
spend = n * pilot_p["spend_per_branch"]

scenarios = {}
for scenario, label in SCENARIOS.items():
    p = params(scenario)
    curve = ltv_customer_year[ltv_customer_year["scenario"] == label]
    last = curve.iloc[-1]
    survive = [p["retention_rate"] ** (y - 1) / (1 + p["discount_rate"]) ** y for y in range(1, YEARS + 1)]
    cross = [s if y >= p["cross_sell_start_year"] else 0.0 for y, s in zip(range(1, YEARS + 1), survive)]
    cumulative = [0.0] + curve["cumulative_ltv_usd"].tolist()
    payback = next((12 * (y - 1 + (spend / incremental - cumulative[y - 1]) / (cumulative[y] - cumulative[y - 1]))
                    for y in range(1, YEARS + 1) if cumulative[y] * incremental >= spend), None)
    customers = float(reachable.sum() * p["adoption_rate"])
    scenarios[label] = {
        "retention_rate": p["retention_rate"], "cross_sell_probability": p["cross_sell_probability"],
        "adoption_rate": p["adoption_rate"], "discount_rate": p["discount_rate"],
        "components": {
            "Debit interchange": round(last["interchange_usd"] * sum(survive)),
            "Deposit spread": round(last["deposit_spread_usd"] * sum(survive)),
            "Fees (kept low)": round(last["fees_usd"] * sum(survive)),
            f"Cross-sell by yr {int(p['cross_sell_start_year'])}": round(p["cross_sell_probability"] * p["cross_sell_annual_revenue"] * sum(cross)),
            "Acquisition cost": -round(p["acquisition_cost"]),
        },
        "ltv_gross": round(last["cumulative_ltv_usd"]), "ltv_net": round(last["cumulative_net_ltv_usd"]),
        "new_customers": round(customers), "footprint_value": round(customers * last["cumulative_net_ltv_usd"]),
        "pilot_return": round(incremental * last["cumulative_ltv_usd"] / spend, 2),
        "pilot_payback_months": None if payback is None else round(payback),
    }

# Tile map: the metro core on a fixed grid of square tiles, shaded by reachable households per tile
# (five equal-count bands). The window is centred on where the reachable households are.
COLS, ROWS, TILE_DEG = 32, 18, 0.05
scale = np.cos(np.radians(np.average(dim_tract["latitude"], weights=reachable)))
size = TILE_DEG
x0 = np.average(dim_tract["longitude"], weights=reachable) * scale - COLS * size / 2
y1 = np.average(dim_tract["latitude"], weights=reachable) + ROWS * size / 2
rows = ROWS
# Each tract's main ZIP code and county, so a tile can say which part of the metro it covers.
zips = read("zcta_tract").sort_values("overlap_land_m2", ascending=False).drop_duplicates("tract_geoid").set_index("tract_geoid")["zcta"]
tiles = pd.DataFrame({"c": np.floor((dim_tract["longitude"] * scale - x0) / size).astype(int),
                      "r": np.floor((y1 - dim_tract["latitude"]) / size).astype(int), "reachable": reachable,
                      "zip": dim_tract["tract_geoid"].map(zips), "county": dim_tract["county_fips"].map(dim_county.set_index("county_fips")["county_name"])})
tiles = tiles[tiles["c"].between(0, COLS - 1) & tiles["r"].between(0, ROWS - 1)]
map_share = tiles["reachable"].sum() / reachable.sum()
heaviest = lambda col: lambda g: list(g.groupby(col)["reachable"].sum().sort_values(ascending=False).index[:3])
tiles = tiles.groupby(["c", "r"])[["reachable", "zip", "county"]].apply(lambda g: pd.Series({
    "reachable": g["reachable"].sum(), "tracts": len(g), "zips": heaviest("zip")(g), "county": heaviest("county")(g)[0]})).reset_index()
tiles["band"] = np.ceil(tiles["reachable"].rank(pct=True) * 5).astype(int)
tile_lat, tile_lon = y1 - (tiles["r"] + 0.5) * size, (x0 + (tiles["c"] + 0.5) * size) / scale
_, tile_km = nearest(tile_lat, tile_lon, local["latitude"], local["longitude"])
tiles["miles_to_local_branch"] = (tile_km / MILES).round(1)
# The city is the one on the nearest branch's address (any bank), which is how FDIC records place.
idx, _ = nearest(tile_lat, tile_lon, dim_branch["latitude"], dim_branch["longitude"])
tiles["city"] = dim_branch["city"].str.title().to_numpy()[idx]
tiles["reachable"] = tiles["reachable"].round().astype(int)
gap = tiles[tiles["band"] == 5].sort_values("miles_to_local_branch").iloc[-1]
place = lambda df: [{"x": x, "y": y, "bank": bank, "office": office.title(), "city": city.title()}
                    for x, y, bank, office, city in zip(((df["longitude"] * scale - x0) / size).round(2), ((y1 - df["latitude"]) / size).round(2),
                                                        df["bank_name"], df["office_name"], df["city"])
                    if 0 <= x <= COLS and 0 <= y <= ROWS]

# Counties: the signal that stands furthest above the metro figure names the top need; the playbook maps it to a plan.
latest = fact_tract_lending["year"].max()
by_county = dim_tract.assign(reachable=reachable).groupby("county_fips").agg(
    households=("households", "sum"), reachable=("reachable", "sum"),
    unbanked=("est_unbanked_households", "sum"), limited_english=("limited_english_households", "sum"))
county_of = dim_tract.set_index("tract_geoid")["county_fips"]
lend = fact_tract_lending[fact_tract_lending["year"] == latest].assign(county_fips=lambda d: d["tract_geoid"].map(county_of))
by_county["denials"] = lend.groupby("county_fips")["denials"].sum()
by_county["applications"] = lend.groupby("county_fips")["applications"].sum()
access = fact_tract_complaints[fact_tract_complaints["product_group"] == "banking_access"]
by_county["complaints"] = access.assign(county_fips=access["tract_geoid"].map(county_of)).groupby("county_fips")["complaints"].sum()
closed = fact_branch_events[fact_branch_events["event"] == "Closing"]
by_county["closings"] = closed.assign(county_fips=closed["tract_geoid"].map(county_of)).groupby("county_fips").size()
by_county = by_county.fillna(0)


def signals(d):
    return pd.DataFrame({
        "unbanked_share": d["unbanked"] / d["households"], "limited_english_share": d["limited_english"] / d["households"],
        "denial_rate": d["denials"] / d["applications"], "complaints_per_1k": d["complaints"] / d["households"] * 1000,
        "closings_per_100k": d["closings"] / d["households"] * 100000})


metro_signals = signals(by_county.sum().to_frame().T).iloc[0]
relative = signals(by_county) / metro_signals
playbook = read("campaign_playbook").set_index("signal")
names = dim_county.set_index("county_fips")["county_name"]
counties = []
for fips in by_county.sort_values("reachable", ascending=False).index[:4]:
    signal = relative.loc[fips, playbook.index].idxmax()
    counties.append({
        "area": names[fips], "signal": signal, "times_metro": round(float(relative.loc[fips, signal]), 2),
        **playbook.loc[signal].to_dict(),
        "value": {label: round(by_county.loc[fips, "reachable"] * params(s)["adoption_rate"] * scenarios[label]["ltv_net"])
                  for s, label in SCENARIOS.items()},
    })

reasons = survey[(survey["geography_type"] == "national") & (survey["year"] == survey["year"].max())
                 & (survey["measure"] == "Main reason unbanked")].set_index("category")["percent"]
responses = read("trust_responses")
events_net = (fact_branch_events["event"] == "Closing").sum() - (fact_branch_events["event"] == "Opening").sum()

site = {
    "region": {"name": read("sample_counties")["cbsa_title"].iloc[0],
               "counties": len(dim_county), "tracts": len(dim_tract), "households": int(households),
               "survey_year": int(survey["year"].max())},
    "reachable_households": round(float(reachable.sum())),
    "reachable_rate": round(float(reachable.sum() / households), 4),
    "unbanked_households": round(float(dim_tract["est_unbanked_households"].sum())),
    "underbanked_households": round(float(dim_tract["est_underbanked_households"].sum())),
    "scenarios": scenarios,
    "map": {"cols": COLS, "rows": rows, "tile_miles": round(TILE_DEG * 69.0, 1), "share_of_reachable": round(float(map_share), 3),
            "tiles": tiles[["c", "r", "band", "city", "county", "zips", "reachable", "tracts", "miles_to_local_branch"]].to_dict("records"),
            "local_branches": place(local[~local["uninum"].isin(pilot_bank["uninum"])]),
            "pilot_branches": place(arms["pilot"]), "control_branches": place(arms["control"]),
            "gap": {"c": int(gap["c"]), "r": int(gap["r"]), "miles": float(gap["miles_to_local_branch"])}},
    "counties": counties,
    "pilot": {
        "bank": pilot_bank["bank_name"].iloc[0], "branches": n, "spend": round(spend),
        "pilot_offices": arms["pilot"]["office_name"].str.title().tolist(),
        "control_offices": arms["control"]["office_name"].str.title().tolist(),
        "catchment_miles": pilot_p["catchment_miles"], "lift": pilot_p["pilot_lift"],
        "catchment_households": {"pilot": round(pilot_reach), "control": round(control_reach)},
        "opened": {"pilot": round(opened["pilot"]), "control": round(opened["control"] * control_scale)},
        "activation": {"pilot": pilot_p["pilot_activation_rate"], "control": pilot_p["control_activation_rate"]},
        "active": {"pilot": round(active["pilot"]), "control": round(active["control"] * control_scale)},
        "incremental_active": round(incremental),
        "new_deposits": round(active["pilot"] * pilot_p["avg_opening_deposit"]),
        "cost_per_incremental_active": round(spend / incremental),
    },
    "trust": {
        "signals": [
            {"label": f"Mortgage denial rate ({latest})", "value": f"{metro_signals['denial_rate']:.0%}"},
            {"label": "Net branch closings since 2021", "value": f"{events_net:,}"},
            {"label": "Banking complaints / 1K households", "value": f"{metro_signals['complaints_per_1k']:.1f}"},
            {"label": "Limited-English households", "value": f"{metro_signals['limited_english_share']:.0%}"},
        ],
        "reasons": [{"reason": r.reason, "percent": round(float(reasons[r.reason])), "response": r.response}
                    for r in responses.itertuples() if r.reason in reasons.index],
    },
}
site_json = json.dumps(site, indent=1)
print(f"website snapshot: {len(site_json) / 1024:.0f} KB")

## Write tables

In Fabric: replace the rows of each Delta table, cast to the column types Create Tables defined.
Locally: write CSVs to `sample-output/`.

In [ ]:
TABLES = {
    "dim_county": dim_county, "dim_tract": dim_tract, "dim_year": dim_year,
    "dim_bank": dim_bank, "dim_branch": dim_branch,
    "fact_branch_deposits": fact_branch_deposits, "fact_branch_events": fact_branch_events,
    "fact_tract_lending": fact_tract_lending, "fact_tract_complaints": fact_tract_complaints,
    "scenario_ltv": scenario_ltv, "tract_ltv": tract_ltv, "ltv_assumptions": ltv_assumptions,
    "ltv_customer_year": ltv_customer_year,
}


def as_text(df: pd.DataFrame) -> pd.DataFrame:
    """Every value as text with None for missing; whole numbers without a trailing .0 so they cast to bigint."""
    out = {}
    for col in df.columns:
        s = df[col]
        if pd.api.types.is_float_dtype(s) and (s.dropna() % 1 == 0).all():
            s = s.astype("Int64")
        out[col] = s.astype("string").astype(object).where(s.notna(), None)
    return pd.DataFrame(out)


def spark_type(s: pd.Series) -> str:
    if pd.api.types.is_bool_dtype(s):
        return "boolean"
    if pd.api.types.is_integer_dtype(s):
        return "bigint"
    return "double" if pd.api.types.is_float_dtype(s) else "string"


if "spark" in globals():
    for name, df in TABLES.items():
        if spark.catalog.tableExists(name):  # noqa: F821 (spark is provided by the Fabric runtime)
            types = {f.name: f.dataType.simpleString() for f in spark.table(name).schema}  # noqa: F821
            if set(types) != set(df.columns):
                types = {col: spark_type(df[col]) for col in df.columns}  # an extra table whose columns changed
        else:
            types = {col: spark_type(df[col]) for col in df.columns}
        text = spark.createDataFrame(as_text(df[list(types)]), schema=", ".join(f"`{c}` string" for c in types))  # noqa: F821
        (
            text.selectExpr(*[f"cast(round(cast(`{c}` as double)) as bigint) as `{c}`" if t == "bigint"
                              else f"cast(`{c}` as {t}) as `{c}`" for c, t in types.items()])
            .write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)
        )
        print(f"wrote {name}: {len(df)} rows")
    (LAKEHOUSE_FILES / "site_data.json").write_text(site_json)
    print("wrote Files/dapa/site_data.json")
else:
    out = Path("sample-output")
    out.mkdir(exist_ok=True)
    for name, df in TABLES.items():
        df.to_csv(out / f"{name}.csv", index=False)
        print(f"wrote {out / name}.csv: {len(df)} rows")
    (out / "site_data.json").write_text(site_json)